In [ ]:
import os
import pandas as pd
import numpy as np
import tqdm
import json
import shutil

# 1. Downloading the data

We download the PPMF data from the U.S. Census Bureau. Note that this might take some time.

**Please replace the path in the following cell with the path of the root of this repository**

In [ ]:
repository_path = 'path/to/desia/repository' # Replace this path

In [ ]:
%cd $repository_path
!mkdir -p ./datasets/ppmf
%cd ./datasets/ppmf
!curl -o 2020-05-27-ppmf.csv https://www2.census.gov/programs-surveys/decennial/2020/program-management/data-product-planning/2010-demonstration-data-products/01-Redistricting_File--PL_94-171/2020-05-27_ppmf/2020-05-27-ppmf.csv

# 2. Loading the data

Loading the data might take some time.

In [ ]:
data = pd.read_csv(f'{repository_path}/datasets/ppmf/2020-05-27-ppmf.csv')

# 3. Processing the data

In [ ]:
data['code'] = data['TABBLKST'].apply(lambda x: str(x).zfill(2)) + \
               data['TABBLKCOU'].apply(lambda x: str(x).zfill(3)) + \
               data['TABTRACTCE'].apply(lambda x: str(x).zfill(6))+ \
               data['TABBLK'].apply(lambda x: str(x).zfill(4))

In [ ]:
data['index'] = data.index

In [ ]:
block_sizes = data.groupby('code')['index'].count()
percentile_99 = int(np.percentile(block_sizes.values, 99) * 10)

In [ ]:
groups = data.groupby('code')
target_dfs = []
for block_id, block_df in tqdm.tqdm(groups):
    if len(block_df) < percentile_99:
        continue
    target_df = block_df[['QAGE', 'QSEX', 'CENRACE', 'CENHISP']].copy()
    target_df.loc[:, 'CENHISP'] = target_df['CENHISP'].apply(lambda x: 2-x)
    target_df.loc[:, 'QSEX'] = target_df['QSEX'].apply(lambda x: 2-x)
    target_df.insert(0, 'TABBLK', np.zeros(len(target_df)))
    target_df['CENRACE'] -= 1
    target_df['TABBLK'] = target_df['TABBLK'].astype(int)
    os.makedirs(f'{repository_path}/datasets/ppmf', exist_ok=True)
    target_dfs.append((block_id, target_df))

In [ ]:
# keep only the 10 smallest blocks larger than the threshold
smallest_blocks_larger_than_threshold = sorted(target_dfs, key=lambda x: len(x[1]))[:10]
for block_id, target_df in smallest_blocks_larger_than_threshold:
    target_df.to_csv(f'{repository_path}/datasets/ppmf/ppmf_{block_id}.csv', index=False)

In [ ]:
# domain
domain = {"TABBLK": 1, "QAGE": 116, "QSEX": 2, "CENRACE": 63, "CENHISP": 2}
os.makedirs(f'{repository_path}/datasets/ppmf/domain', exist_ok=True)
for block_id, target_df in smallest_blocks_larger_than_threshold:
    with open(f'{repository_path}/datasets/ppmf/domain/ppmf_{block_id}-domain.json', 'w') as fp:
        json.dump(domain, fp)

In [ ]:
# queries
os.makedirs(f'{repository_path}/datasets/ppmf/queries', exist_ok=True)
for block_id, target_df in smallest_blocks_larger_than_threshold:
    shutil.copyfile('block_queries.pkl', f'{repository_path}/datasets/ppmf/queries/ppmf_{block_id}-set.pkl')